# 04 - Evaluation

Objectif : evaluer le modele sur le test.
- Accuracy test
- Courbes accuracy/loss
- Matrice de confusion + rapport

Fonction testee : `evaluer_modele`

In [ ]:
import sys
from pathlib import Path
import pickle
import tensorflow as tf

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import chargement_pretraitement_dataset, preparer_donnees
from src.model_architecture import conception_modele_cnn, compiler_modele
from src.train import entrainer_modele
from src.evaluate import evaluer_modele

print("Imports OK")

In [ ]:
# 1. Preparer les donnees + entrainer rapidement (ou charger un modele existant)
DATA_PATH = PROJECT_ROOT / "data" / "raw"
X, y = chargement_pretraitement_dataset(str(DATA_PATH))
X_train, X_test, y_train, y_test, le, class_weights = preparer_donnees(X, y)

# Option A : charger un modele deja entraine
model_path = PROJECT_ROOT / "models" / "best_model.keras"
if model_path.exists():
    print(f"Chargement {model_path}")
    model = tf.keras.models.load_model(str(model_path))
    # Il faut aussi entrainer un peu pour avoir 'history' -> sinon on entraine 2 epochs pour la demo
    # Pour une vraie eval, remplace par un entrainement complet du notebook 03
    history = None  # on va faire un mini-fit pour les courbes si besoin
else:
    print("Pas de modele trouve, entrainement rapide (3 epochs)...")
    from src.model_architecture import conception_modele_cnn, compiler_modele
    model = conception_modele_cnn(nb_classes=len(le.classes_))
    model = compiler_modele(model)
    history = entrainer_modele(model, X_train, y_train, X_test, y_test, class_weights, epochs=3, batch_size=32)

print("Pret pour evaluation")

In [ ]:
# 2. Si on a charge le modele (history=None), on refait un mini entrainement pour avoir les courbes
# Sinon on evalue directement
if history is None:
    # Petit entrainement juste pour generer les courbes
    from src.model_architecture import conception_modele_cnn, compiler_modele
    from src.train import entrainer_modele
    # On entraine 3 epochs juste pour demo des courbes
    history = entrainer_modele(model, X_train, y_train, X_test, y_test, class_weights, epochs=3, batch_size=32)

result = evaluer_modele(model, history, X_test, y_test, le)
print(f"\nAccuracy : {result['accuracy']*100:.2f}%")

### Astuce pour evaluer un vrai modele
1. Lance d'abord `03_model_training.ipynb` avec `epochs=30`
2. Puis dans ce notebook remplace tout par :
```python
import tensorflow as tf
model = tf.keras.models.load_model("../models/best_model.keras")
import pickle
le = pickle.load(open("../models/label_encoder.pkl", "rb"))
# puis evaluer_modele(...)
```